## 1. Setup & Upload Data

In [1]:
# Install dependencies (Colab usually has these already, but this ensures compatibility)
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
# Upload your dataset
from google.colab import files
import io

uploaded = files.upload()  # Choose your CSV file (e.g. DS_332_Lab_Final_Dataset...csv)
INPUT_FILE = list(uploaded.keys())[0]
print(f"Loaded file: {INPUT_FILE}")


ModuleNotFoundError: No module named 'google.colab'

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer

TARGET_COL = "Chance of Admit"
RANDOM_STATE = 42

sns.set_theme(style="whitegrid")


## 2. Load Data

In [5]:
df = pd.read_csv(/Users/jayanto/Downloads/lab_final_data.csv)
df.columns = [c.strip() for c in df.columns]
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


SyntaxError: invalid syntax (4112474971.py, line 1)

## 3. Clean Data

In [ ]:
print("Missing values per column (before cleaning):")
print(df.isna().sum())


In [ ]:
df_clean = df.copy()

# Drop rows where the target itself is missing (can't use for supervised learning)
df_clean = df_clean.dropna(subset=[TARGET_COL])

# Fill missing categorical values (Degree) with the mode
if "Degree" in df_clean.columns:
    df_clean["Degree"] = df_clean["Degree"].fillna(df_clean["Degree"].mode()[0])

# Numeric columns to impute with median
numeric_cols = ["GRE Score", "TOEFL Score", "University Rating", "SOP", "LOR", "CGPA", "Research"]
numeric_cols = [c for c in numeric_cols if c in df_clean.columns]

imputer = SimpleImputer(strategy="median")
df_clean[numeric_cols] = imputer.fit_transform(df_clean[numeric_cols])

# Research is binary (0/1) - round after imputation
if "Research" in df_clean.columns:
    df_clean["Research"] = df_clean["Research"].round().astype(int)

print("Missing values per column (after cleaning):")
print(df_clean.isna().sum())
print(f"\nRows remaining after cleaning: {len(df_clean)}")
df_clean.head()


In [ ]:
# Save cleaned dataset
df_clean.to_csv("cleaned_dataset.csv", index=False)
print("Saved cleaned dataset to cleaned_dataset.csv")

# Download it (optional)
from google.colab import files as colab_files
colab_files.download("cleaned_dataset.csv")


## 4. Exploratory Data Analysis

In [ ]:
print("SUMMARY STATISTICS")
df_clean.describe(include="all")


In [ ]:
# Correlation heatmap
numeric_df = df_clean.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of target variable
plt.figure(figsize=(8, 5))
sns.histplot(df_clean[TARGET_COL], kde=True, bins=20)
plt.title("Distribution of Chance of Admit")
plt.xlabel("Chance of Admit")
plt.tight_layout()
plt.show()


In [ ]:
# Chance of Admit by Degree
if "Degree" in df_clean.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df_clean, x="Degree", y=TARGET_COL)
    plt.title("Chance of Admit by Degree")
    plt.tight_layout()
    plt.show()


In [ ]:
# CGPA vs Chance of Admit
if "CGPA" in df_clean.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_clean, x="CGPA", y=TARGET_COL,
                     hue="Research" if "Research" in df_clean.columns else None)
    plt.title("CGPA vs Chance of Admit")
    plt.tight_layout()
    plt.show()


In [ ]:
# Pairplot of key numeric features
key_cols = [c for c in ["GRE Score", "TOEFL Score", "CGPA", TARGET_COL] if c in df_clean.columns]
if len(key_cols) >= 2:
    sns.pairplot(df_clean[key_cols])
    plt.show()


## 5. Modeling

In [ ]:
df_model = df_clean.copy()

# One-hot encode Degree if present
if "Degree" in df_model.columns:
    df_model = pd.get_dummies(df_model, columns=["Degree"], drop_first=True)

drop_cols = [c for c in ["Serial No.", TARGET_COL] if c in df_model.columns]
X = df_model.drop(columns=drop_cols)
y = df_model[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R2:   {r2:.4f}")

pd.DataFrame(results).T


In [ ]:
# Feature importance from Random Forest
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Random Forest Feature Importances:")
print(importances)

plt.figure(figsize=(8, 5))
importances.plot(kind="barh")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
